In [16]:
import os
import json

cwd = os.getcwd()
dir = os.path.dirname(cwd)
reference_path = os.path.join(dir, "data", "referenceMIDI.json")
response_path = os.path.join(dir, "data", "responseMIDI.json")

with open(reference_path) as f1:
    reference = json.load(f1)

with open(response_path) as f2:
    response = json.load(f2)

# Dynamic Time Warping

The goal of DTW is to find an optimal possibly nonlinear alignment between response MIDI sequence to reference MIDI sequence.

Basic approach:
- Evaluating the local cost measure for each pair of elements in the response(X) and reference(Y) sequences. 
- Dynamic programming to find an alignment path between X and Y having minimal overall cost, i.e. DTW distance. The algorithm computes a cumulative distance path, the timestamps of the target MIDI are warped so they perfectly align with the anchor points of the reference MIDI.

In [ ]:
def compute_cost(note1, note2):
    """
    Compute the local cost measure for each pair of notes.
 
    Only pitch is involved in the cost calculation, since the purpose 
    is to check if the user played missing or extra notes.
 
    Args:
        note1: dict with keys "pitch" (int), "start" (float), "duration" (float)
        note2: dict with keys "pitch" (int), "start" (float), "duration" (float)
 
    Returns:
        int: cost value >= 0 (lower means more similar pitch)
    """
    return int(abs(note1["pitch"] - note2["pitch"]))


import numpy as np
def compute_cost_matrix(response_notes, ref_notes):
    """
    Build the local cost matrix C of size (N x M).
    C[i, j] = note_cost(ref_notes[i], response_notes[j])
 
    Args:
        response_notes: list of dicts, each with keys "pitch", "start", "duration"
        ref_notes: list of dicts, each with keys "pitch", "start", "duration"
 
    Returns:
        numpy array of shape (N, M) - cost matrix where C[i,j] is the cost of 
        aligning response_notes[i] with ref_notes[j]
    """
    N = len(response_notes)
    M = len(ref_notes)
    C= np.zeros((N, M))
 
    for i in range(N):
        for j in range(M):
            C[i, j] = compute_cost(response_notes[i], ref_notes[j])
    return C


def accumulate_cost_matrix(C):
    """
    Build the accumulated cost matrix D of size (N+1 x M+1) 
    using small trick for simplifying the initialization:
    set:
        D[0, 0] = 0
        D[n, 0] = inf  for n >= 1
        D[0, m] = inf  for m >= 1
    for all n in [1..N] and m in [1..M]:
        D[i, j] = C[i, j] + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    
    Args:
        numpy array of shape (N, M) — the local cost matrix
 
    Returns:
        numpy array of shape (N+1, M+1) — the accumulated cost matrix
    """
    N, M = C.shape
    D = np.full((N + 1, M + 1), np.inf)
    D[0, 0] = 0.0
 
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            D[i, j] = C[i - 1, j - 1] + min(
                D[i - 1, j], # vertical step
                D[i, j - 1], # horizontal step
                D[i - 1, j - 1]) # diagonal step
 
    return D

def backtrack_warping_path(D):
    """
    Backtrack through the D to find the optimal warping path P.
 
    Args:
        D: numpy array of shape (N+1, M+1) — the accumulated cost matrix
 
    Returns:
        list of tuples — the optimal warping path as a list of (i, j) indices
        ordered from the start (0, 0) to the end (N-1, M-1)
    """
    N = D.shape[0] - 1
    M = D.shape[1] - 1

    path = []
 
    while N > 0 and M > 0:
        path.append((N-1, M-1))
        # Find the minimum cost step
        min_step = min(D[N - 1, M], D[N, M - 1], D[N - 1, M - 1])
        if min_step == D[N - 1, M]: # vertical step
            N -= 1
        elif min_step == D[N, M - 1]: # horizontal step
            M -= 1
        else: # diagonal step
            N -= 1
            M -= 1
 
    # Reverse to get the path from start to end
    path.reverse()  
    return path


def note_alignment_DTW(responseNotes, refNotes):
    """
    DTW algorithm based on dynamic programming.
    To find the optimal alignment between response and reference MIDI notes.
    """
    pass

Reference:

M. Müller, Fundamentals of Music Processing (Chpater 3.2 Dynamic Time Warping). Cham: Springer International Publishing, 2021, ISBN: 9783030698072. DOI:https://doi.org/10.1007/978-3-030-69808-9.
